# 🏭 Iron Ore Quality Predictor
### Predicting % Silica Concentrate in Mining Flotation Plant

**Dataset:** Mining Process Flotation Plant Database  
**Target:** `% Silica Concentrate` (lower = better iron ore purity)  
**Features:** 22 process variables (air flows, levels, pH, density, etc.)  
**Model:** Random Forest Regressor + XGBoost  

---

## 📦 Step 1: Install & Import Libraries

In [ ]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn joblib -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import xgboost as xgb
import joblib
import os

print('✅ All libraries imported successfully!')
print(f'XGBoost version: {xgb.__version__}')

## 📂 Step 2: Mount Google Drive & Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Upload your CSV via the Files panel OR place it in Google Drive
# Option A: Upload directly to Colab session
from google.colab import files
print('📁 If you have not uploaded yet, run the cell below to upload your CSV')

In [ ]:
# Upload the CSV file
uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0]
print(f'File uploaded: {DATA_PATH}')

In [ ]:
# Load dataset
df = pd.read_csv(DATA_PATH, decimal=',')

print(f'✅ Dataset loaded!')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print()
df.head()

## 🔍 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print('=== Dataset Info ===')
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')
print(f'Missing values: {df.isnull().sum().sum()}')
print()
print('=== Data Types ===')
print(df.dtypes)

In [ ]:
# Target variable distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['% Silica Concentrate'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution of % Silica Concentrate', fontsize=14, fontweight='bold')
axes[0].set_xlabel('% Silica Concentrate')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['% Silica Concentrate'].mean(), color='red', linestyle='--', label=f'Mean: {df["% Silica Concentrate"].mean():.2f}%')
axes[0].legend()

axes[1].hist(df['% Iron Concentrate'], bins=50, color='darkorange', edgecolor='white', alpha=0.8)
axes[1].set_title('Distribution of % Iron Concentrate', fontsize=14, fontweight='bold')
axes[1].set_xlabel('% Iron Concentrate')
axes[1].set_ylabel('Frequency')
axes[1].axvline(df['% Iron Concentrate'].mean(), color='red', linestyle='--', label=f'Mean: {df["% Iron Concentrate"].mean():.2f}%')
axes[1].legend()

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: target_distribution.png')

In [ ]:
# Correlation heatmap
numeric_df = df.drop(columns=['date'])
corr = numeric_df.corr()

plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0,
            linewidths=0.5, fmt='.2f', square=True)
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: correlation_heatmap.png')

In [ ]:
# Top correlations with target
target_corr = corr['% Silica Concentrate'].drop('% Silica Concentrate').abs().sort_values(ascending=False)
print('=== Top 10 Features Correlated with % Silica Concentrate ===')
print(target_corr.head(10).to_string())

plt.figure(figsize=(10, 6))
target_corr.head(10).plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Top 10 Feature Correlations with % Silica Concentrate', fontsize=13, fontweight='bold')
plt.xlabel('Absolute Correlation')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## 🛠️ Step 4: Data Preprocessing

In [ ]:
# Drop date column, separate features and target
df_clean = df.drop(columns=['date']).copy()

# Sample for faster training (use full data if you have time/GPU)
SAMPLE_SIZE = 100000  # ~100K rows for efficient training
df_sampled = df_clean.sample(n=min(SAMPLE_SIZE, len(df_clean)), random_state=42)
print(f'Using {len(df_sampled):,} rows for training')

# Features and target
TARGET = '% Silica Concentrate'
FEATURES = [col for col in df_sampled.columns if col != TARGET]

X = df_sampled[FEATURES].values
y = df_sampled[TARGET].values

print(f'Features: {len(FEATURES)}')
print(f'Target: {TARGET}')
print(f'X shape: {X.shape}, y shape: {y.shape}')

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training set: {X_train.shape[0]:,} samples')
print(f'Test set:     {X_test.shape[0]:,} samples')
print('✅ Preprocessing complete!')

## 🤖 Step 5: Train Models

In [ ]:
# ── Model 1: Random Forest ──
print('Training Random Forest...')
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    n_jobs=-1,
    random_state=42
)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_mae  = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2   = r2_score(y_test, rf_pred)

print(f'Random Forest → MAE: {rf_mae:.4f} | RMSE: {rf_rmse:.4f} | R²: {rf_r2:.4f}')

In [ ]:
# ── Model 2: XGBoost ──
print('Training XGBoost...')
xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method='hist',
    verbosity=0
)
xgb_model.fit(X_train_scaled, y_train,
              eval_set=[(X_test_scaled, y_test)],
              verbose=50)
xgb_pred = xgb_model.predict(X_test_scaled)

xgb_mae  = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_r2   = r2_score(y_test, xgb_pred)

print(f'XGBoost        → MAE: {xgb_mae:.4f} | RMSE: {xgb_rmse:.4f} | R²: {xgb_r2:.4f}')

## 📊 Step 6: Model Evaluation & Visualization

In [ ]:
# Comparison table
results = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost'],
    'MAE':   [rf_mae,  xgb_mae],
    'RMSE':  [rf_rmse, xgb_rmse],
    'R²':    [rf_r2,   xgb_r2]
})
print('=== Model Comparison ===')
print(results.to_string(index=False))

best_model_name = 'XGBoost' if xgb_r2 > rf_r2 else 'Random Forest'
best_pred = xgb_pred if xgb_r2 > rf_r2 else rf_pred
print(f'\n🏆 Best Model: {best_model_name} (R² = {max(xgb_r2, rf_r2):.4f})')

In [ ]:
# Actual vs Predicted plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, pred, name, color in zip(
    axes,
    [rf_pred, xgb_pred],
    ['Random Forest', 'XGBoost'],
    ['steelblue', 'darkorange']
):
    ax.scatter(y_test[:2000], pred[:2000], alpha=0.3, s=8, color=color)
    lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect Prediction')
    ax.set_xlabel('Actual % Silica Concentrate')
    ax.set_ylabel('Predicted % Silica Concentrate')
    ax.set_title(f'{name}\nR² = {r2_score(y_test, pred):.4f}', fontweight='bold')
    ax.legend()

plt.suptitle('Actual vs Predicted — % Silica Concentrate', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: actual_vs_predicted.png')

In [ ]:
# Feature Importance (Random Forest)
fi = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values(ascending=False)

plt.figure(figsize=(10, 7))
fi.head(15).plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Top 15 Feature Importances (Random Forest)', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: feature_importance.png')
print()
print('Top 5 Important Features:')
print(fi.head(5).to_string())

In [ ]:
# Residuals plot
residuals = y_test - best_pred
plt.figure(figsize=(10, 4))
plt.scatter(best_pred[:3000], residuals[:3000], alpha=0.3, s=8, color='purple')
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted % Silica Concentrate')
plt.ylabel('Residuals')
plt.title(f'Residual Plot — {best_model_name}', fontweight='bold')
plt.tight_layout()
plt.savefig('residuals_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: residuals_plot.png')

## 💾 Step 7: Save Model & Scaler

In [ ]:
os.makedirs('models', exist_ok=True)

# Save both models
joblib.dump(rf_model,  'models/random_forest_model.pkl')
joblib.dump(xgb_model, 'models/xgboost_model.pkl')
joblib.dump(scaler,    'models/scaler.pkl')
joblib.dump(FEATURES,  'models/feature_names.pkl')

print('✅ Models saved!')
print('  models/random_forest_model.pkl')
print('  models/xgboost_model.pkl')
print('  models/scaler.pkl')
print('  models/feature_names.pkl')

## 🚀 Step 8: Quick Prediction Demo

In [ ]:
# Load and use the model
loaded_rf  = joblib.load('models/random_forest_model.pkl')
loaded_xgb = joblib.load('models/xgboost_model.pkl')
loaded_scaler = joblib.load('models/scaler.pkl')
loaded_features = joblib.load('models/feature_names.pkl')

# Sample prediction using first row of test data
sample = X_test[0:1]
sample_scaled = loaded_scaler.transform(sample)

pred_rf  = loaded_rf.predict(sample)[0]
pred_xgb = loaded_xgb.predict(sample_scaled)[0]
actual   = y_test[0]

print('=== Sample Prediction ===')
print(f'Actual % Silica Concentrate:        {actual:.4f}%')
print(f'Random Forest Prediction:           {pred_rf:.4f}%')
print(f'XGBoost Prediction:                 {pred_xgb:.4f}%')
print(f'RF Error:  {abs(actual - pred_rf):.4f}%')
print(f'XGB Error: {abs(actual - pred_xgb):.4f}%')

## 📋 Step 9: Summary Report

In [ ]:
print('=' * 55)
print('       IRON ORE QUALITY PREDICTOR — SUMMARY')
print('=' * 55)
print(f'Dataset:    Mining Flotation Plant Database')
print(f'Rows used:  {len(df_sampled):,} (of {len(df):,} total)')
print(f'Features:   {len(FEATURES)}')
print(f'Target:     % Silica Concentrate')
print()
print('Model Performance:')
print(f'  Random Forest → R²: {rf_r2:.4f} | MAE: {rf_mae:.4f}')
print(f'  XGBoost        → R²: {xgb_r2:.4f} | MAE: {xgb_mae:.4f}')
print()
print(f'🏆 Best Model: {best_model_name}')
print()
print('Saved Files:')
print('  models/random_forest_model.pkl')
print('  models/xgboost_model.pkl')
print('  models/scaler.pkl')
print('  target_distribution.png')
print('  correlation_heatmap.png')
print('  actual_vs_predicted.png')
print('  feature_importance.png')
print('=' * 55)